# 🧠 Step 5c: 1D Vision Transformer with Overlapping Patches & 1D Augmentation (500 Hz)

This notebook implements advanced deep learning enhancements for the **500 Hz high-resolution PTB-XL ECG dataset** (5,000 samples per lead) using the **4 selected unique leads** (`aVF`, `III`, `I`, `II`):

1. **1D ECG Data Augmentation**:
   - Random Amplitude Scaling ($\pm 15\%$ gain variations)
   - Random Time Shift (phase jitter $\pm 100$ samples)
   - Random Gaussian Noise (low-amplitude white noise)
   - Random Temporal Cutout (brief zero-masking)

2. **Overlapping Patch Tokenization**:
   - Standard ViT cuts signals distinctly (`stride = patch_size`), causing boundary artifacts.
   - **Overlapping ViT** sets `patch_size = 100` samples (0.2s duration) and `stride = 50` samples (**50% overlap**).
   - Number of tokens: $N_p = \frac{5000 - 100}{50} + 1 = 99$ overlapping tokens.
   - Smooth transitions preserve continuous P-QRS-T complex transitions across patch borders!

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, roc_curve, auc

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Using compute device: {device}")

In [ ]:
# 1D ECG Data Augmentation Classes
class ECGAugmenter1D:
    def __init__(self, p=0.5):
        self.p = p
    def __call__(self, x):
        # x: (channels, seq_len)
        if torch.rand(1).item() < self.p:
            scale = torch.empty(1).uniform_(0.85, 1.15).item()
            x = x * scale
        if torch.rand(1).item() < self.p:
            shift = torch.randint(-100, 101, (1,)).item()
            x = torch.roll(x, shifts=shift, dims=-1)
        if torch.rand(1).item() < self.p:
            x = x + torch.randn_like(x) * 0.02
        if torch.rand(1).item() < self.p:
            hole = torch.randint(10, 100, (1,)).item()
            start = torch.randint(0, x.shape[-1] - hole, (1,)).item()
            x = x.clone()
            x[:, start:start+hole] = 0.0
        return x

augmenter = ECGAugmenter1D(p=0.5)
print("1D ECG Augmenter initialized successfully!")

In [ ]:
# Overlapping Patch 1D Vision Transformer Architecture
class OverlappingPatchEmbedding1D(nn.Module):
    def __init__(self, in_channels=4, patch_size=100, stride=50, embed_dim=128):
        super().__init__()
        self.proj = nn.Conv1d(in_channels, embed_dim, kernel_size=patch_size, stride=stride, bias=False)
        self.norm = nn.LayerNorm(embed_dim)
    def forward(self, x):
        x = self.proj(x).transpose(1, 2) # (B, num_patches, embed_dim)
        return self.norm(x)

class ECG_ViT1D_Overlapping(nn.Module):
    def __init__(self, in_channels=4, seq_len=5000, patch_size=100, patch_stride=50, embed_dim=128, depth=4, num_heads=4, num_classes=5):
        super().__init__()
        self.num_patches = (seq_len - patch_size) // patch_stride + 1
        self.patch_embed = OverlappingPatchEmbedding1D(in_channels, patch_size, patch_stride, embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(0.1)

        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim*2, dropout=0.1, activation="gelu", batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(embed_dim, num_classes))

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1) + self.pos_embed
        x = self.pos_drop(x)
        x = self.transformer(x)
        features = self.norm(x[:, 0])
        return self.head(features)

model = ECG_ViT1D_Overlapping(in_channels=4, seq_len=5000, patch_size=100, patch_stride=50).to(device)
print(f"Initialized Overlapping ViT for 500 Hz: {model.num_patches} patches, {sum(p.numel() for p in model.parameters()):,} parameters.")

In [ ]:
# Visual Plot: Demonstrate 1D Data Augmentation on a Sample ECG
sample_signal = torch.sin(torch.linspace(0, 10*np.pi, 5000)).unsqueeze(0).repeat(4, 1) # (4, 5000)
aug_signal = augmenter(sample_signal)

plt.figure(figsize=(12, 5))
plt.plot(sample_signal[0].numpy()[:500], label="Original 500 Hz Signal", color="#1d3557", lw=1.5)
plt.plot(aug_signal[0].numpy()[:500], label="1D Augmented Signal (Scale + Jitter + Noise)", color="#e63946", lw=1.2, linestyle="--")
plt.title("1D ECG Signal Data Augmentation Demonstration", fontsize=12, fontweight="bold")
plt.xlabel("Time Samples (500 Hz)")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True, alpha=0.5)
plt.show()